# 02 — Limpeza da base de solicitações

Este notebook continua o código da aula, mantendo a estrutura original e acrescentando as verificações pedidas no desafio.

In [ ]:
import pandas as pd

#AULA 1 DE PANDAS - SUJEIRA

# Carregar a base de dados
df = pd.read_csv(
    "https://raw.githubusercontent.com/Projetar-Solucoes/materiais-trilha/main/semana-04/solicitacoes_semana4.csv"
)

# Questão 1: Tamanho da base (linhas e colunas)
print("QUESTÃO 1 - TAMANHO DA BASE")
print(f"Linhas e colunas: {df.shape}")
print(f"Quantidade de linhas: {df.shape[0]}")
print(f"Quantidade de colunas: {df.shape[1]}")

# Guardar o tamanho original para comparar depois
linhas_iniciais = df.shape[0]

# Questão 2: Tipos das colunas
print("\nQUESTÃO 2 - TIPOS DAS COLUNAS")
print(df.dtypes)

# Questão 3: Valores vazios por coluna
print("\nQUESTÃO 3 - VALORES VAZIOS POR COLUNA")
print(df.isna().sum())

# Valores vazios especificamente na coluna prioridade
print(f"Valores vazios em prioridade: {df['prioridade'].isna().sum()}")

# Questão 4: Linhas repetidas
print("\nQUESTÃO 4 - LINHAS REPETIDAS")
print(f"Quantidade de linhas repetidas: {df.duplicated().sum()}")

# Questão 5: Valores diferentes em categoria, prioridade e status
print("\nQUESTÃO 5 - VALORES DIFERENTES")
print(f"Categorias diferentes: {df['categoria'].nunique()}")
print(f"Valores de categoria: {df['categoria'].unique()}")
print(f"\nPrioridades diferentes: {df['prioridade'].nunique()}")
print(f"Valores de prioridade: {df['prioridade'].unique()}")
print(f"\nStatus diferentes: {df['status'].nunique()}")
print(f"Valores de status: {df['status'].unique()}")

# Desafio extra: quantidade de solicitações por unidade
print("\nDESAFIO EXTRA - SOLICITAÇÕES POR UNIDADE")
print(df['unidade'].value_counts())


## Aula 2 de Pandas — limpeza

In [ ]:
#AULA 2 DE PANDAS - LIMPEZA
print("LIMPEZA.. \n")

# Linhas repetidas
repetidas = int(df.duplicated().sum())
df = df.drop_duplicates()                  # tira as linhas repetidas
print(f"Quantidade de linhas: {df.shape[0]}")

# deixa a categoria num padrão só
categoria_antes = df["categoria"].copy()
df["categoria"] = df["categoria"].str.strip().str.lower()
padrao = {
    "suporte ti": "Suporte TI",
    "suporte de ti": "Suporte TI",
    "acesso": "Acesso",
    "financeiro": "Financeiro",
    "infraestrutura": "Infraestrutura",
    "infra": "Infraestrutura",
    "dúvida": "Dúvida",
    "duvida": "Dúvida",
}
df["categoria"] = df["categoria"].map(padrao)

print(f"Quantas solicitações de dúvidas?: {df['categoria'].value_counts().get('Dúvida', 0)}")
print(f"Quantas solicitações de Suporte de TI?: {df['categoria'].value_counts().get('Suporte TI', 0)}")

# Conferir categorias que não foram reconhecidas pelo dicionário
print(f"Categorias vazias após padronização: {df['categoria'].isna().sum()}")

# Padronizar status
status_antes = df["status"].copy()
df["status"] = df["status"].str.strip().str.lower().str.capitalize()
print("\nValores de status após a limpeza:")
print(df["status"].value_counts(dropna=False))

# Padronizar prioridade, incluindo os valores conhecidos
prioridade_antes = df["prioridade"].copy()
df["prioridade"] = df["prioridade"].str.strip().str.lower()
padrao_prioridade = {
    "baixo": "Baixa",
    "baixa": "Baixa",
    "medio": "Média",
    "médio": "Média",
    "media": "Média",
    "média": "Média",
    "alto": "Alta",
    "alta": "Alta",
    "urgente": "Urgente",
}
df["prioridade"] = df["prioridade"].map(padrao_prioridade)
print("\nValores de prioridade após a limpeza:")
print(df["prioridade"].value_counts(dropna=False))

# Avaliação: só notas entre 1 e 5 entram na média
df["avaliacao"] = pd.to_numeric(df["avaliacao"], errors="coerce")
avaliacoes_validas = df["avaliacao"].between(1, 5)
avaliacoes_invalidas = int((~avaliacoes_validas & df["avaliacao"].notna()).sum())
df.loc[~avaliacoes_validas, "avaliacao"] = pd.NA
print(f"Avaliações fora do intervalo de 1 a 5: {avaliacoes_invalidas}")

print("\nQuantidade por categoria:")
print(df["categoria"].value_counts(dropna=False))


## Quantidade e média de avaliação por categoria

As médias consideram somente notas numéricas de 1 a 5. Categorias vazias após o mapeamento são mantidas como `NaN` nesta versão para que você consiga identificar quais valores do CSV não estavam no dicionário `padrao`.

In [ ]:
# Quantidade de solicitações por categoria
quantidade_categoria = df["categoria"].value_counts(dropna=False).sort_index(key=lambda x: x.astype(str))

# Média de avaliação por categoria (somente notas de 1 a 5)
media_avaliacao = df.groupby("categoria", dropna=False)["avaliacao"].mean().round(2)

resumo_categoria = pd.DataFrame({
    "quantidade_solicitacoes": quantidade_categoria,
    "media_avaliacao": media_avaliacao,
})
print("RESUMO POR CATEGORIA")
display(resumo_categoria)

# Conferir se a soma das quantidades é igual ao total de linhas limpas
total_por_categoria = int(quantidade_categoria.sum())
total_base_limpa = len(df)
print(f"Soma das quantidades por categoria: {total_por_categoria}")
print(f"Total de linhas da base limpa: {total_base_limpa}")
assert total_por_categoria == total_base_limpa
print("Conferência OK: os totais são iguais.")


## Gerar `relatorio_qualidade.md`

A célula abaixo conta as linhas afetadas e grava o relatório na mesma pasta em que o notebook estiver sendo executado.

In [ ]:
# Quantidade de linhas afetadas por cada padronização
# Para categoria, considera linhas que mudaram de texto ou viraram NaN após o mapa.
categoria_afetadas = int((categoria_antes.fillna("").astype(str).str.strip() != df["categoria"].fillna("").astype(str)).sum())
status_afetadas = int((status_antes.fillna("").astype(str).str.strip() != df["status"].fillna("").astype(str)).sum())
prioridade_afetadas = int((prioridade_antes.fillna("").astype(str).str.strip() != df["prioridade"].fillna("").astype(str)).sum())

relatorio = f'''# Relatório de qualidade — base de solicitações

| Problema | Linhas afetadas | O que eu fiz |
|---|---:|---|
| Linhas repetidas | {repetidas} | Removi com `drop_duplicates()` |
| Categoria escrita de jeitos diferentes ou não reconhecida | {categoria_afetadas} | Padronizei usando o dicionário `padrao`; valores não incluídos no dicionário ficam vazios para revisão |
| Status escrito de jeitos diferentes | {status_afetadas} | Removi espaços e padronizei para inicial maiúscula |
| Prioridade escrita de jeitos diferentes ou não reconhecida | {prioridade_afetadas} | Padronizei os níveis conhecidos usando `padrao_prioridade`; valores não reconhecidos ficam vazios para revisão |
| Avaliação fora do intervalo de 1 a 5 | {avaliacoes_invalidas} | Desconsiderei essas notas no cálculo das médias |

**Resultado:** a base tinha {linhas_iniciais} linhas antes da limpeza e ficou com {len(df)} linhas após a remoção das duplicatas. A soma das quantidades por categoria foi {total_por_categoria}, igual ao total da base limpa ({total_base_limpa}).
'''

with open("relatorio_qualidade.md", "w", encoding="utf-8") as arquivo:
    arquivo.write(relatorio)

print(relatorio)
print("Relatório salvo em relatorio_qualidade.md")
